# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [1]:
%matplotlib ipympl

In [2]:
from analysis.build_data import load_trace_df
from analysis.calibration import (
    category_table,
    expected_calibration_error,
    kalshibench_metrics,
)


In [3]:
# one scoreable forecast per row, from every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name="sweep-20261106-1744")

## Dataset statistics

Table 2 of the paper: the markets in the sample by category, with the share resolving to the priced outcome (see `category_table` in `analysis/calibration.py`).

In [54]:
category_table(forecasts).round(1)

,N,%,Yes%
category,,,
politics,984,50.8,23.5
finance,452,23.3,44.2
economics,448,23.1,41.3
elections,42,2.2,28.6
companies,11,0.6,45.5


## Expected calibration error

Forecasts binned by stated confidence (`confidence_bin`, from `load_trace_df`; see `expected_calibration_error` in `analysis/calibration.py`).

In [ ]:
ece = expected_calibration_error(forecasts)

In [ ]:
ece_by_model = forecasts.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

## KalshiBench table

Table 4 of the paper, per model (see `kalshibench_metrics` in `analysis/calibration.py` for the columns).

In [4]:
model_table = forecasts.groupby("model").apply(
    kalshibench_metrics, include_groups=False
)

In [5]:
model_table.round(3)

,N,Acc,F1,F1yes,Brier,BSS,ECE,MCE
model,,,,,,,,
claude-fable-5-1,727.0,70.839,0.618,0.433,0.184,0.171,0.038,0.093
claude-opus-4-5,997.0,69.208,0.628,0.473,0.210,0.038,0.046,0.350
claude-opus-5-5,1904.0,71.376,0.626,0.446,0.185,0.175,0.030,0.047


In [ ]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = forecasts.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)